# NeSy-KG trên Flora (zero-shot skeleton)

Baseline là **Flora** (CVPR 2026 Findings). Phần NeSy dùng KG primitive một cách mềm ở phía ngữ nghĩa, không khớp luật và không cần ground primitive trên skeleton:

- `neighbors`: độ giống chọn láng giềng = `(1-α)·CLIP + α·KG` (độ giống theo primitive chung của hai lớp).
- `compose`: cộng `β·ngữ nghĩa lớp ghép từ mô tả primitive` vào `attuned_semantics`.
- `both`: dùng cả hai. `none`: Flora gốc.

Code: `zeroshot/Flora/kg_semantics.py`, các chỗ sửa trong `trainer.py` (`load_prototypes`) và `options.py` (`--kg-mode`, `--kg-alpha`, `--kg-beta`, `--kg-forbid-weight`).

**Trước khi chạy**: phải commit và push `zeroshot/Flora` (có `kg_semantics.py`, `kg/`, `semantics/ntu*_llm.txt`, `trainer.py`, `options.py`). Notebook clone repo từ GitHub nên chỉ thấy phần đã push. Repo private thì lưu token vào Colab Secrets với tên `GH_TOKEN`.

KG chỉ được viết từ tên lớp và từ vựng primitive, không dùng dữ liệu skeleton.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, shutil, subprocess, time, re
import numpy as np, pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/BaoHuy/zeroshot'
FEAT_ARCHIVE = f'{DRIVE_DIR}/flora_skeleton_features.archive'   # dùng chung cache với flora_repro.ipynb
OUT_DIR = f'{DRIVE_DIR}/flora_nesy'                             # log + bảng kết quả
EXTRACT = '/content/flora_features'
os.makedirs(OUT_DIR, exist_ok=True)

REPO = '/content/thesis'
if not os.path.exists(REPO):
    try:
        from google.colab import userdata
        tok = userdata.get('GH_TOKEN')
    except Exception:
        tok = None
    url = f'https://{tok}@github.com/NguyenBaoHuy05/thesis.git' if tok else 'https://github.com/NguyenBaoHuy05/thesis.git'
    subprocess.run(['git', 'clone', '-q', '--depth', '1', url, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
!git -C {REPO} log -1 --format='%h %s'
subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'], check=True, stdout=open(f'{OUT_DIR}/thesis_commit.txt', 'w'))

FLORA = f'{REPO}/zeroshot/Flora'
need = ['kg_semantics.py', 'kg/primitive_v2.json', 'kg/kg_rules_v2.json', 'semantics/ntu60_llm.txt', 'semantics/ntu120_llm.txt']
missing = [p for p in need if not os.path.exists(f'{FLORA}/{p}')]
assert not missing, f'Repo chưa có {missing}: commit và push zeroshot/Flora trước'
assert '--kg-mode' in open(f'{FLORA}/options.py').read(), 'options.py trên GitHub chưa có --kg-mode: chưa push bản sửa trainer/options'
for n in (60, 120):
    n_lines = len(open(f'{FLORA}/semantics/ntu{n}_llm.txt', encoding='utf-8').read().splitlines())
    assert n_lines == n, f'ntu{n}_llm.txt có {n_lines} dòng, cần {n}'
print('repo OK')

In [ ]:
!pip -q install gdown transformers pyyaml pandas

# models/fm.py luôn import Shift-GCN, và Shift-GCN import CUDA op `shift_cuda`.
# Vì use_features=True nên Shift-GCN không bao giờ chạy, chỉ cần một module rỗng để import qua.
stub = f'{FLORA}/models/shiftgcn/Temporal_shift/cuda/shift_cuda.py'
if not os.path.exists(stub):
    with open(stub, 'w') as f:
        f.write('# stub: Shift-GCN CUDA op is never called when use_features=True\n')

!cd {FLORA} && python -c "from models.fm import MappingNet; import kg_semantics; print('import OK')"

## Tải feature đã trích sẵn

Link Google Drive lấy từ README của Flora. Nếu `gdown` báo lỗi quota: mở `https://drive.google.com/file/d/1wrxetP_ItfJfxAfNPLdCRC2FZ8Mb9Pa1/view`, chọn *Make a copy* vào Drive của bạn, rồi đổi tên/di chuyển file thành `FEAT_ARCHIVE` ở trên. Nếu đã chạy `flora_repro.ipynb` thì file này có sẵn.

In [ ]:
FEAT_ID = '1wrxetP_ItfJfxAfNPLdCRC2FZ8Mb9Pa1'
if not os.path.exists(FEAT_ARCHIVE):
    !gdown {FEAT_ID} -O {FEAT_ARCHIVE}
!ls -lh {FEAT_ARCHIVE}

def find_star():
    hits = glob.glob(f'{EXTRACT}/**/star_features', recursive=True)
    return hits[0] if hits else None

if find_star() is None:
    os.makedirs(EXTRACT, exist_ok=True)
    kind = subprocess.run(['file', '-b', FEAT_ARCHIVE], capture_output=True, text=True).stdout
    print('archive type:', kind)
    if 'Zip' in kind:
        subprocess.run(['unzip', '-q', FEAT_ARCHIVE, '-d', EXTRACT], check=True)
    elif '7-zip' in kind:
        subprocess.run(['7z', 'x', '-y', f'-o{EXTRACT}', FEAT_ARCHIVE], check=True, stdout=subprocess.DEVNULL)
    elif 'RAR' in kind:
        subprocess.run(['unrar', 'x', '-o+', FEAT_ARCHIVE, EXTRACT + '/'], check=True, stdout=subprocess.DEVNULL)
    else:  # tar / tar.gz
        subprocess.run(['tar', '-xf', FEAT_ARCHIVE, '-C', EXTRACT], check=True)

STAR_DIR = find_star()
assert STAR_DIR, 'Không thấy thư mục star_features sau khi giải nén'
os.makedirs(f'{FLORA}/skeleton_features', exist_ok=True)   # thư mục này nằm trong .gitignore của Flora
link = f'{FLORA}/skeleton_features/star_features'
if not os.path.exists(link):
    os.symlink(STAR_DIR, link)
!ls -lh {link}/

## Kiểm tra KG (không cần GPU)

In số primitive mỗi lớp, độ giống KG giữa các lớp, và các cặp lớp mà KG **không phân biệt được** (hàng giống hệt nhau). Các cặp đó (mặc/cởi áo, giày, kính, mũ) thành láng giềng mạnh của nhau ở chế độ `neighbors`.

In [ ]:
!cd {FLORA} && python kg_semantics.py 60
!cd {FLORA} && python kg_semantics.py 120 | head -8

## Sinh config và chạy

Mỗi run: `align_iter=1000` iteration cho phase 1, rồi `total_iter=1200` cho phase 2, đánh giá mỗi 10 iteration. Log ghi vào `OUT_DIR/<kg>/<task>_ZSL_seed<k>/run.log`. Run nào đã xong sẽ bị bỏ qua, nên Colab ngắt thì chạy lại cell là được.

Siêu tham số theo split lấy từ config gốc của Flora (SynSE hoặc STAR), chỉ thêm `kg_mode` và các tham số KG.

In [ ]:
import yaml
os.makedirs(f'{FLORA}/configs/repro', exist_ok=True)

# task -> (config gốc để lấy siêu tham số theo split, ZSL top-1 bài báo Flora báo cáo trên feature STAR)
SPLITS = {
    'ntu60_xsub_seen55_unseen5':    ('configs/star/ntu60_xsub_unseen5.yaml',    85.8),
    'ntu60_xsub_seen48_unseen12':   ('configs/synse/ntu60_xsub_unseen12.yaml',  61.5),
    'ntu120_xsub_seen110_unseen10': ('configs/synse/ntu120_xsub_unseen10.yaml', 80.7),
    'ntu120_xsub_seen96_unseen24':  ('configs/synse/ntu120_xsub_unseen24.yaml', 64.1),
}
KG_MODES = ['none', 'neighbors', 'compose', 'both']
FEAT = 'star'   # feature gốc của Flora (1s-Shift-GCN)

def kg_label(kg, ov=None):
    ov = ov or {}
    return 'none' if kg == 'none' else f"kg_{kg}_a{ov.get('kg_alpha', 0.5)}_b{ov.get('kg_beta', 0.3)}"

def out_dir(task, seed, kg='none', ov=None):
    return f'{OUT_DIR}/{kg_label(kg, ov)}/{task}_ZSL_seed{seed}'

def make_cfg(task, seed=1, kg='none', **overrides):
    with open(f'{FLORA}/{SPLITS[task][0]}') as f:
        cfg = yaml.load(f, Loader=yaml.FullLoader)
    out = out_dir(task, seed, kg, overrides)
    cfg.update(
        task_name=task,
        data_path=f'skeleton_features/{FEAT}_features/{task}.npz',
        use_features=True,
        shiftgcn_checkpoint_path=None,  # không load Shift-GCN
        clip_checkpoint_path='openai/clip-vit-large-patch14-336',
        num_worker=2,
        setting='ZSL',
        seed=seed,
        out_dir=out,
        kg_mode=kg,
    )
    cfg.update(overrides)  # ví dụ kg_alpha, kg_beta
    path = f'{FLORA}/configs/repro/{task}_ZSL_seed{seed}_{kg_label(kg, overrides)}.yaml'
    with open(path, 'w') as f:
        yaml.dump(cfg, f, sort_keys=False)
    return path, out

def run(task, seed=1, kg='none', **overrides):
    path, out = make_cfg(task, seed, kg, **overrides)
    log = f'{out}/run.log'
    if os.path.exists(log) and 'Best' in open(log).read():
        print(f'[skip] {kg} {task} seed{seed} đã xong')
        return
    shutil.rmtree(out, ignore_errors=True)  # logger mở file ở chế độ append, xóa để không lẫn run cũ
    os.makedirs(os.path.dirname(out), exist_ok=True)
    t0 = time.time()
    p = subprocess.run(['python', 'main.py', '--config', path], cwd=FLORA, capture_output=True, text=True)
    print(f'[{kg} {task} seed{seed}] exit={p.returncode}  {(time.time()-t0)/60:.1f} phút  {overrides or ""}')
    print('\n'.join(p.stdout.strip().splitlines()[-2:]))
    if p.returncode != 0:
        print(p.stderr[-3000:])

def has_feat(task):
    return os.path.exists(f'{FLORA}/skeleton_features/{FEAT}_features/{task}.npz')

In [ ]:
# Bước 1: chạy thử chế độ `both` một seed trên 48/12, rồi xác nhận log có dòng KG bật.
# Nếu lỗi thì thường ở load_prototypes (kích thước embedding) hoặc đường dẫn kg/.
T48 = 'ntu60_xsub_seen48_unseen12'
run(T48, seed=1, kg='both')
!grep -E "KG (neighbors|compose)|Attuned|Best" {out_dir(T48, 1, 'both')}/run.log

In [ ]:
# Bước 2: bốn chế độ x 3 seed. 'none' là Flora gốc cùng feature, cùng seed, nên dùng làm mốc cặp theo seed.
SEEDS = [1, 2, 3]
KG_TASKS = [T48]   # thêm 'ntu120_xsub_seen96_unseen24' nếu còn thời gian
for task in KG_TASKS:
    for seed in SEEDS:
        for kg in KG_MODES:
            if has_feat(task):
                run(task, seed=seed, kg=kg)

# Độ nhạy (tuỳ chọn, chỉ để xem, KHÔNG chọn siêu tham số theo kết quả này vì đó là chọn trên unseen test):
# for a in (0.25, 0.5, 0.75):
#     run(T48, seed=1, kg='neighbors', kg_alpha=a)

## Tổng hợp

- `last` = ZSL ở lần đánh giá cuối (iter 1200), không chọn trên test. Đây là số dùng để so.
- `best` = max qua mọi lần đánh giá, là cách bài báo Flora báo cáo, chọn trên unseen test nên lạc quan. Chỉ để đối chiếu với bài báo.
- `delta` = `last` của chế độ KG trừ `last` của `none` cùng seed. Cần cả `none` và chế độ đó chạy xong cùng seed.

In [ ]:
ZSL_RE = re.compile(r'Iter (\d+): ZSL Accuracy: ([\d.]+)%')

def read_log(task, seed, kg):
    log = f'{out_dir(task, seed, kg)}/run.log'
    if not os.path.exists(log):
        return None
    rows = [tuple(float(x) for x in m.groups()) for m in ZSL_RE.finditer(open(log).read())]
    return np.array(rows) if rows else None

rows = []
for kg in KG_MODES:
    for task, (_, paper) in SPLITS.items():
        for seed in SEEDS:
            a = read_log(task, seed, kg)
            if a is not None:
                acc = a[:, 1]
                rows.append(dict(split=task, kg=kg, seed=seed, paper=paper, best=acc.max(), last=acc[-1],
                                 last10_mean=acc[-10:].mean(), last10_std=acc[-10:].std(), n_evals=len(acc)))
df = pd.DataFrame(rows).round(2)
df.to_csv(f'{OUT_DIR}/summary_zsl.csv', index=False)
display(df)

if len(df):
    agg = df.groupby(['split', 'kg'], sort=False).agg(
        n_seeds=('seed', 'count'), paper=('paper', 'first'),
        last_mean=('last', 'mean'), last_std=('last', 'std'),
        best_mean=('best', 'mean'), best_std=('best', 'std')).round(2)
    agg.to_csv(f'{OUT_DIR}/summary_zsl_seeds.csv')
    display(agg)

    piv = df.pivot_table(index=['split', 'seed'], columns='kg', values='last')
    if 'none' in piv:
        delta = piv.drop(columns='none').sub(piv['none'], axis=0)
        d = delta.groupby('split').agg(['mean', 'std', 'count']).round(2)
        d.to_csv(f'{OUT_DIR}/delta_vs_none.csv')
        display(d)

## Đọc kết quả

- **`none` khớp Flora gốc** (trong khoảng 1-2 điểm của số bài báo ở `best`) thì baseline sạch, các chênh lệch bên dưới là do KG.
- **delta trung bình nhỏ so với độ lệch chuẩn qua seed** thì chưa kết luận được KG có ích. Chạy thêm seed trước khi tin.
- **`neighbors` kém `none`** có thể do các cặp lớp KG không phân biệt được (mặc/cởi). Thử giảm `kg_alpha` hoặc thêm primitive chiều (mặc lên, cởi ra) vào KG.
- **`compose` kém `none`** thì thử giảm `kg_beta`. Ghép hai chuỗi token CLIP khác nhau theo vị trí là xấp xỉ thô, cùng kiểu với cách Flora trộn láng giềng.
- Mọi số ở đây là ZSL trên đúng split Flora. GZSL chưa chạy; `--kg-mode` dùng được cho GZSL (đổi `setting`), nhưng mỗi lần đánh giá GZSL tốn gấp 25-145 lần ZSL.